In [1]:
# Read the metadata extracted by notebook 01 - no Oracle connection needed
import json
from pathlib import Path

import yaml

# project root = parent of notebook/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

# metadata written by notebook 01
METADATA_DIR = PROJECT_ROOT / "metadata" / "text2sql"
CATALOG_PATH = PROJECT_ROOT / "catalog" / "catalog.yaml"
manifest = json.loads((METADATA_DIR / "manifest.json").read_text(encoding="utf-8"))
print(f"schemas found: {list(manifest['schemas'])}")

schemas found: ['CO', 'HR', 'SH']


In [2]:
# Load each schema's metadata into a dict keyed by schema name
schema_data = {}
for schema, info in manifest["schemas"].items():
    schema_data[schema] = json.loads((METADATA_DIR / info["file"]).read_text(encoding="utf-8"))
    print(f"{schema}: {info['object_count']} objects")

CO: 11 objects
HR: 8 objects
SH: 12 objects


In [3]:
# Curated business meaning per workspace (one workspace = one schema).
# Edit these by hand - they are what the intent agent reads to route a question.
WORKSPACE_DESCRIPTIONS = {
    "CO": "Customer orders, order items, shipments, stores, inventory, products and product reviews",
    "HR": "Employees, departments, jobs, locations, regions and job history",
    "SH": "Sales facts with product, customer, time, channel and promotion dimensions, plus profit and monthly-sales aggregations",
}

In [4]:
# Check for cross-schema foreign keys first: verified FKs are the strongest
# evidence for cross-schema joins. The POLICY dict itself is computed in a
# later cell (after the candidate connections are built) because the join
# mode depends on BOTH signals: verified FKs and candidate connections.
cross_schema_fks = [
    (schema, name, fk)
    for schema, data in schema_data.items()
    for name, obj in data["objects"].items()
    for fk in obj["foreign_keys"]
    if fk["references_schema"] != schema
]
print(f"cross-schema foreign keys: {len(cross_schema_fks)}")
for schema, name, fk in cross_schema_fks:
    print(f"  {schema}.{name} -> {fk['references_schema']}.{fk['references_table']}")

cross-schema foreign keys: 0


In [5]:
# Auto-generate the workspace structure from the metadata.
# Columns keep name/type/nullable/description; FKs become explicit join hints.
def build_workspace(data):
    tables = {}
    for name, obj in data["objects"].items():
        tables[name] = {
            "object_type": obj["object_type"],
            "description": obj["description"],
            "columns": [
                {
                    "name": c["name"],
                    "type": c["type"],
                    "nullable": c["nullable"],
                    "description": c["description"],
                }
                for c in obj["columns"]
            ],
            "primary_key": obj["primary_key"],
            "primary_key_name": obj["primary_key_name"],
            "unique_constraints": obj["unique_constraints"],
            "foreign_keys": [
                {
                    "name": fk["name"],
                    "columns": fk["columns"],
                    "references_schema": fk["references_schema"],
                    "references_table": fk["references_table"],
                    "references_columns": fk["references_columns"],
                    "reference_resolved": fk["reference_resolved"],
                }
                for fk in obj["foreign_keys"]
            ],
        }
    return tables

workspaces = {}
for schema, data in schema_data.items():
    workspaces[schema] = {
        "description": WORKSPACE_DESCRIPTIONS[schema],
        "tables": build_workspace(data),
    }
    print(f"{schema}: {len(workspaces[schema]['tables'])} tables in workspace")

CO: 11 tables in workspace
HR: 8 tables in workspace
SH: 12 tables in workspace


In [6]:
# Examples are optional and OFF for now (zero-shot experiment).
# If any are added later they must be written in analyst language
# (business questions, no table/schema/database names), e.g.
#   {"question": "How much did we sell last month", "sql": ...}
for ws in workspaces.values():
    ws["examples"] = []
print("examples: none (zero-shot mode)")

examples: none (zero-shot mode)


In [7]:
# Candidate schema-to-schema connections: tables with the same name in
# different schemas. A shared column (same name AND same type) is recorded
# as a possible join key; otherwise the key is unresolved. These are
# informational only - cross-schema joins stay disallowed in the policy.
connections = []
by_name = {}
for ws_name, ws in workspaces.items():
    for tname in ws["tables"]:
        by_name.setdefault(tname, []).append(ws_name)

for tname, ws_names in sorted(by_name.items()):
    if len(ws_names) < 2:
        continue
    # a possible join key is a column name present on every side with the same type
    col_maps = [
        {c["name"]: c["type"] for c in workspaces[w]["tables"][tname]["columns"]}
        for w in ws_names
    ]
    join_key = "unresolved"
    for cname, ctype in col_maps[0].items():
        if all(m.get(cname) == ctype for m in col_maps[1:]):
            join_key = cname
            break
    connections.append({
        "tables": [f"{w}.{tname}" for w in ws_names],
        "join_key": join_key,
        "status": "candidate",
    })
    print(f"connection: {[f'{w}.{tname}' for w in ws_names]} join_key={join_key}")
print(f"connections found: {len(connections)}")

connection: ['HR.COUNTRIES', 'SH.COUNTRIES'] join_key=COUNTRY_NAME
connection: ['CO.CUSTOMERS', 'SH.CUSTOMERS'] join_key=unresolved
connection: ['CO.PRODUCTS', 'SH.PRODUCTS'] join_key=unresolved
connections found: 3


In [8]:
# Query policy, derived from the data - never hardcoded assumptions:
# - verified cross-schema FKs exist  -> joins allowed via those FKs only
# - else candidate connections exist -> joins allowed via catalog connections
#   only (the join key is disclosed to the model as a candidate, never invented)
# - neither                          -> joins disallowed, questions stay in
#                                       one workspace
# max_workspaces_per_question lets the intent agent (notebook 05) route a
# question to TWO workspaces when the catalog records a connection between
# them. No approval workflow: the catalog data is the gate.
# DE DECISION (documented): candidate connections are recorded below but are
# NOT trusted as join evidence - in practice the pairs share no usable keys
# (CO<->SH: different key names and zero name overlap; HR<->SH: name-match
# only). Cross-schema joins are therefore disabled until verified FKs exist.
# Flip to False (or delete this line) when verified cross-schema FKs appear.
CROSS_SCHEMA_DISABLED = True

POLICY = {
    "max_workspaces_per_question": 1 if CROSS_SCHEMA_DISABLED else (
        2 if (cross_schema_fks or connections) else 1),
    "cross_schema_joins": "disallowed" if CROSS_SCHEMA_DISABLED else (
        "verified_fk_only" if cross_schema_fks
        else "catalog_connections_only" if connections
        else "disallowed"
    ),
    "cross_schema_fk_count": len(cross_schema_fks),
    "connection_count": len(connections),
}
print(f"policy: {POLICY}")

policy: {'max_workspaces_per_question': 1, 'cross_schema_joins': 'disallowed', 'cross_schema_fk_count': 0, 'connection_count': 3}


In [9]:
# Assemble and write the catalog. Each workspace stays self-contained;
# connections are recorded separately and are not used while policy
# disallows cross-schema joins.
catalog = {
    "catalog_version": "1.2",
    "database": manifest["database"],
    "policy": POLICY,
    "workspaces": workspaces,
    "connections": connections,
}
CATALOG_PATH.parent.mkdir(parents=True, exist_ok=True)
# NOTE for developers: this file is GENERATED - comments cannot survive a
# re-run, so the authoritative explanations live in this notebook and in
# catalog/rules.yaml (hand-edited). The header below is written every time.
catalog_header = """# QueryLLM catalog - GENERATED by notebook/02_catalog.ipynb, do not edit by hand.
# Structure:
#   policy      - query routing rules (max workspaces per question, cross-schema mode)
#   workspaces  - one per schema, self-contained: tables + columns + PK + FK
#   connections - candidate cross-schema links from same-named tables;
#                 join_key is the shared column, or 'unresolved' when the
#                 schemas use different key names (no reliable join yet)
# Cross-schema rule: joins are allowed ONLY through the connections listed
#   here. The intent agent (05) may route to 2 workspaces only when they are
#   connected, and the SQL writer (03) only uses the recorded join key.
"""
CATALOG_PATH.write_text(
    catalog_header + yaml.safe_dump(catalog, sort_keys=False, allow_unicode=True, width=100),
    encoding="utf-8",
)
print(f"catalog written -> {CATALOG_PATH}")

catalog written -> /workspace/project/queryllm/catalog/catalog.yaml


In [10]:
# Validate the catalog against the metadata: every table, column and FK target
# must exist. Later agents trust this file, so catch drift here.
errors = []
for ws_name, ws in workspaces.items():
    meta_tables = schema_data[ws_name]["objects"]
    for table_name, table in ws["tables"].items():
        meta_cols = {c["name"] for c in meta_tables[table_name]["columns"]}
        cat_cols = {c["name"] for c in table["columns"]}
        if cat_cols != meta_cols:
            errors.append(f"{ws_name}.{table_name}: column set differs from metadata")
        for fk in table["foreign_keys"]:
            if not fk["reference_resolved"]:
                errors.append(f"{ws_name}.{table_name}: FK {fk['name']} unresolved")
            elif fk["references_table"] not in meta_tables:
                errors.append(f"{ws_name}.{table_name}: FK target {fk['references_table']} not in metadata")
    if not ws.get("examples"):
        print(f"note: {ws_name} has no examples (zero-shot mode)")

for conn in connections:
    for qualified in conn["tables"]:
        ws_name, tname = qualified.split(".", 1)
        if tname not in workspaces[ws_name]["tables"]:
            errors.append(f"connection {qualified}: table not in metadata")

if errors:
    for e in errors:
        print("ERROR:", e)
    raise SystemExit(1)

total_tables = sum(len(ws["tables"]) for ws in workspaces.values())
print(f"catalog valid: {len(workspaces)} workspaces, {total_tables} tables, "
      f"{len(connections)} candidate connections")

note: CO has no examples (zero-shot mode)
note: HR has no examples (zero-shot mode)
note: SH has no examples (zero-shot mode)
catalog valid: 3 workspaces, 31 tables, 3 candidate connections
